In [1]:
!pip install tensorrt --quiet

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 16.6 MB/s eta 0:00:0000:010:01


In [2]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No CUDA GPU found")

True
Tesla T4


In [3]:
!git clone https://github.com/FaridRash/brain-ct-hemorrhage-segmentation.git

Cloning into 'brain-ct-hemorrhage-segmentation'...
remote: Enumerating objects: 577, done.
remote: Counting objects: 100% (227/227), done.
remote: Compressing objects: 100% (178/178), done.
remote: Total 577 (delta 128), reused 113 (delta 46), pack-reused 350 (from 1)
Receiving objects: 100% (577/577), 614.59 MiB | 25.47 MiB/s, done.
Resolving deltas: 100% (254/254), done.
Updating files: 100% (208/208), done.


In [8]:
import shutil, os

src = "/kaggle/working/brain-ct-hemorrhage-segmentation/Output/ONNX/2026-06-23"
dst = "/kaggle/working"

for f in ["unet_brain_ct.onnx", "unet_brain_ct.onnx.data"]:
    shutil.copy(f"{src}/{f}", f"{dst}/{f}")
    print(f"Copied: {f}")

# Verify
for f in ["unet_brain_ct.onnx", "unet_brain_ct.onnx.data"]:
    size = os.path.getsize(f"{dst}/{f}")
    print(f"{f}: {size/1024:.1f} KB")

Copied: unet_brain_ct.onnx
Copied: unet_brain_ct.onnx.data
unet_brain_ct.onnx: 12.8 KB
unet_brain_ct.onnx.data: 7552.0 KB


In [9]:
import onnx

model = onnx.load("/kaggle/working/unet_brain_ct.onnx")
onnx.checker.check_model(model)
print("ONNX model is valid.")
print(f"Opset version: {model.opset_import[0].version}")

ONNX model is valid.
Opset version: 17


In [10]:
import tensorrt as trt
print(f"TensorRT version: {trt.__version__}")

TensorRT version: 11.1.0.106


In [18]:
import tensorrt as trt
import os

ONNX_PATH = "/kaggle/working/unet_brain_ct.onnx"
ENGINE_PATH = "/kaggle/working/unet_brain_ct.trt"

TRT_LOGGER = trt.Logger(trt.Logger.WARNING)

def build_engine(onnx_path):
    builder = trt.Builder(TRT_LOGGER)
    network = builder.create_network()
    parser = trt.OnnxParser(network, TRT_LOGGER)

    with open(onnx_path, "rb") as f:
        if not parser.parse(f.read()):
            for i in range(parser.num_errors):
                print(f"Parse error {i}: {parser.get_error(i)}")
            raise RuntimeError("ONNX parsing failed.")

    config = builder.create_builder_config()
    config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)

    # Define optimization profile for dynamic batch dimension
    profile = builder.create_optimization_profile()
    profile.set_shape(
        "input",
        min=(1, 3, 128, 128),   # minimum batch size
        opt=(8, 3, 128, 128),   # optimal batch size
        max=(16, 3, 128, 128)   # maximum batch size
    )
    config.add_optimization_profile(profile)

    print("Building engine... (this may take 1-2 minutes)")
    engine = builder.build_serialized_network(network, config)

    if engine is None:
        raise RuntimeError("Engine build failed.")

    with open(ENGINE_PATH, "wb") as f:
        f.write(engine)

    print(f"Engine saved to: {ENGINE_PATH}")
    print(f"Engine size: {os.path.getsize(ENGINE_PATH)/1024:.1f} KB")

build_engine(ONNX_PATH)

[06/25/2026-07:46:35] [TRT] [W] WARNING The logger passed into createInferBuilder differs from one already registered for an existing builder, runtime, or refitter. So the current new logger is ignored, and TensorRT will use the existing one which is returned by nvinfer1::getLogger() instead.
Building engine... (this may take 1-2 minutes)
Engine saved to: /kaggle/working/unet_brain_ct.trt
Engine size: 12956.4 KB


In [19]:
import tensorrt as trt
import numpy as np
import torch

ENGINE_PATH = "/kaggle/working/unet_brain_ct.trt"
TRT_LOGGER = trt.Logger(trt.Logger.WARNING)

# Load engine
with open(ENGINE_PATH, "rb") as f:
    runtime = trt.Runtime(TRT_LOGGER)
    engine = runtime.deserialize_cuda_engine(f.read())

context = engine.create_execution_context()

# Create a dummy input — same shape as your model expects
dummy_input = np.random.rand(1, 3, 128, 128).astype(np.float32)

# Set input shape for dynamic batch
context.set_input_shape("input", dummy_input.shape)

# Allocate output buffer
output = np.empty((1, 1, 128, 128), dtype=np.float32)

# Move to GPU and run
import pycuda.driver as cuda
import pycuda.autoinit

d_input = cuda.mem_alloc(dummy_input.nbytes)
d_output = cuda.mem_alloc(output.nbytes)

cuda.memcpy_htod(d_input, dummy_input)
context.execute_v2([int(d_input), int(d_output)])
cuda.memcpy_dtoh(output, d_output)

print(f"Input shape:  {dummy_input.shape}")
print(f"Output shape: {output.shape}")
print(f"Output range: [{output.min():.4f}, {output.max():.4f}]")
print("TensorRT inference successful.")

[06/25/2026-07:47:29] [TRT] [W] WARNING The logger passed into createInferRuntime differs from one already registered for an existing builder, runtime, or refitter. So the current new logger is ignored, and TensorRT will use the existing one which is returned by nvinfer1::getLogger() instead.
Input shape:  (1, 3, 128, 128)
Output shape: (1, 1, 128, 128)
Output range: [-10.4691, -5.3740]
TensorRT inference successful.


In [24]:
!pip install onnxruntime -q

In [25]:
import onnxruntime as ort

ONNX_PATH = "/kaggle/working/unet_brain_ct.onnx"

# Use the same dummy input
test_input = np.random.rand(1, 3, 128, 128).astype(np.float32)

# --- ONNX inference ---
sess = ort.InferenceSession(ONNX_PATH, providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
onnx_out = sess.run(None, {"input": test_input})[0]

# --- TensorRT inference ---
context.set_input_shape("input", test_input.shape)
trt_out = np.empty((1, 1, 128, 128), dtype=np.float32)

d_input = cuda.mem_alloc(test_input.nbytes)
d_output = cuda.mem_alloc(trt_out.nbytes)

cuda.memcpy_htod(d_input, test_input)
context.execute_v2([int(d_input), int(d_output)])
cuda.memcpy_dtoh(trt_out, d_output)

# --- Compare ---
max_diff = np.abs(onnx_out - trt_out).max()
mean_diff = np.abs(onnx_out - trt_out).mean()

print(f"ONNX  output range: [{onnx_out.min():.4f}, {onnx_out.max():.4f}]")
print(f"TRT   output range: [{trt_out.min():.4f}, {trt_out.max():.4f}]")
print(f"Max diff:  {max_diff:.6f}")
print(f"Mean diff: {mean_diff:.6f}")
print(f"Numerically equivalent: {max_diff < 1e-3}")

ONNX  output range: [-10.5217, -4.8897]
TRT   output range: [-10.5217, -4.8897]
Max diff:  0.000004
Mean diff: 0.000001
Numerically equivalent: True


/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:147: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
